In [2]:
# Standard library
import os
import random
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Torchvision
import torchvision
import torchvision.transforms as transforms

# Progress bars
from tqdm import tqdm

# Experiment tracking
import wandb


plt.style.use("default")
import sys
from dataclasses import dataclass

In [3]:
# 1. Clone your clean repository code from GitHub
REPO_URL = "https://github.com/K0NSTANT1N3/Facial-Expression-Recognition.git"
REPO_NAME = "Facial-Expression-Recognition"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repository already exists. Pulling latest updates...")
    os.system(f"cd {REPO_NAME} && git pull")

# 2. THE SECRET SAUCE: Add your cloned repo directory to Python's path
sys.path.append(os.path.abspath(REPO_NAME))

print("Successfully linked GitHub repository and adjusted system paths!")


Cloning repository...


Cloning into 'Facial-Expression-Recognition'...


Successfully linked GitHub repository and adjusted system paths!


In [4]:
import wandb

wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kende23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import wandb

run = wandb.init(
    project="fer2013",
    entity="kende23-n-a",
    name="test_run"
)

for epoch in range(5):
    run.log({
        "epoch": epoch,
        "accuracy": epoch * 0.1,
        "loss": 1/(epoch+1)
    })

run.finish()

accuracy,▁▃▅▆█
epoch,▁▃▅▆█
loss,█▄▂▁▁
accuracy,0.4
epoch,4
loss,0.2


In [6]:
@dataclass
class Config:
    csv_path: str = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 10

    num_classes: int = 7
    image_size: int = 48

    random_seed: int = 42


config = Config()

In [7]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.random_seed)

In [8]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [9]:
df = pd.read_csv(config.csv_path)

print(df.shape)
df.head()

(28709, 2)


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


In [10]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=config.random_seed,
    stratify=df["emotion"]
)

print(len(train_df))
print(len(val_df))

22967
5742


In [11]:
def pixels_to_image(pixel_string):
    pixels = np.fromstring(
        pixel_string,
        dtype=np.uint8,
        sep=" "
    )

    return pixels.reshape(48, 48)

In [12]:
class FERDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = pixels_to_image(row["pixels"])

        image = image.astype(np.float32) / 255.0

        image = torch.tensor(image).unsqueeze(0)

        label = torch.tensor(
            row["emotion"],
            dtype=torch.long
        )

        return image, label

In [13]:
train_dataset = FERDataset(train_df)
val_dataset = FERDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [14]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 1, 48, 48])
torch.Size([64])


In [15]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),

            nn.Linear(128, 7)
        )

    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(x, start_dim=1)

        x = self.classifier(x)

        return x

In [16]:
model = SimpleCNN().to(device)

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=4608, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=7, bias=True)
  )
)


In [17]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)

Input shape : torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])


In [18]:
criterion = nn.CrossEntropyLoss()

In [19]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.lr
)

In [20]:
wandb.init(
    project="fer2013",
    entity="kende23-n-a",
    name="baseline_cnn",
    config=vars(config)
)

wandb.watch(model)

In [21]:
def train_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() #?

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [22]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [23]:
print(next(model.parameters()).device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

cuda:0
True
Tesla T4


In [24]:
best_val_acc = 0

for epoch in range(config.epochs):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch+1}/{config.epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_baseline_model.pt"
        )

Epoch 1/10 | Train Loss: 1.6898 | Train Acc: 0.3251 | Val Loss: 1.6063 | Val Acc: 0.3795
Epoch 2/10 | Train Loss: 1.5404 | Train Acc: 0.4083 | Val Loss: 1.5295 | Val Acc: 0.4136
Epoch 3/10 | Train Loss: 1.4468 | Train Acc: 0.4456 | Val Loss: 1.4087 | Val Acc: 0.4648
Epoch 4/10 | Train Loss: 1.3618 | Train Acc: 0.4808 | Val Loss: 1.3611 | Val Acc: 0.4815
Epoch 5/10 | Train Loss: 1.2915 | Train Acc: 0.5070 | Val Loss: 1.3389 | Val Acc: 0.4873
Epoch 6/10 | Train Loss: 1.2219 | Train Acc: 0.5390 | Val Loss: 1.3338 | Val Acc: 0.4920
Epoch 7/10 | Train Loss: 1.1535 | Train Acc: 0.5670 | Val Loss: 1.2937 | Val Acc: 0.5017
Epoch 8/10 | Train Loss: 1.0883 | Train Acc: 0.5963 | Val Loss: 1.2835 | Val Acc: 0.5207
Epoch 9/10 | Train Loss: 1.0242 | Train Acc: 0.6239 | Val Loss: 1.3110 | Val Acc: 0.5063
Epoch 10/10 | Train Loss: 0.9546 | Train Acc: 0.6491 | Val Loss: 1.3309 | Val Acc: 0.5183


In [25]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▄▄▅▆▆▇▇█
train_loss,█▇▆▅▄▄▃▂▂▁
val_acc,▁▃▅▆▆▇▇█▇█
val_loss,█▆▄▃▂▂▁▁▂▂
epoch,10
train_acc,0.64915
train_loss,0.95461
val_acc,0.51829
val_loss,1.33088
